# FracAtlas model comparison

This notebook trains three independent image classifiers on the same FracAtlas split. The candidate used by the FractureCare API is selected by **fracture-focused macro recall** (the average recall of ONE_FRACTURE and MULTIPLE_FRACTURES), with all-class macro F1 as the tie-breaker. This prioritises finding the two minority fracture categories in this imbalanced dataset; balanced accuracy, per-class metrics and the confusion matrix are also reported.

| Model | Why it is included |
| --- | --- |
| Custom CNN | A transparent, lightweight baseline that can be changed easily and establishes a reference point. |
| MobileNetV2 | An efficient transfer-learning model suited to limited data and practical CPU inference. |
| EfficientNetB0 | A stronger accuracy/efficiency candidate whose compound scaling often performs well on image classification. |

This comparison is an engineering experiment, not clinical validation.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
SERVICE_DIR = next(path for path in candidates if (path / 'app').is_dir() and (path / 'requirements.txt').exists())
PROJECT_DIR = SERVICE_DIR.parent
sys.path.insert(0, str(SERVICE_DIR))
from app.config import ARTIFACT_DIR, DATASET_CSV, IMAGE_DIR, IMAGE_SIZE, SEED
from app.data import load_manifest
from app.labels import CLASS_NAMES
from app.metrics import SELECTION_METRIC, calculate_classification_metrics, select_best_result
from app.model import MODEL_NAMES, build_model, build_transfer_model

MODEL_EPOCHS = 15
BATCH_SIZE = 32
USE_IMAGENET_WEIGHTS = True  # Set to False for an offline comparison.
MODEL_DIR = ARTIFACT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Dataset:', DATASET_CSV)

I0000 00:00:1787654810.165551    1778 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787654810.259084    1778 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787654811.727207    1778 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0
Dataset: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/Dataset/FracAtlas/dataset.csv


In [2]:
frame = load_manifest()
train, holdout = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=frame['label_index'])
validation, test = train_test_split(holdout, test_size=0.5, random_state=SEED, stratify=holdout['label_index'])
train, validation, test = [part.reset_index(drop=True) for part in (train, validation, test)]
print(f'Images: {len(frame):,} | train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
display(frame['label'].value_counts().reindex(CLASS_NAMES).rename('count').to_frame())

W0000 00:00:1787654843.793304    1778 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1787654843.796944    1778 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1787654843.987826    1778 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4783 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
E0000 00:00:1787654886.640712    1778 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 414/454
W0000 00:00:1787654886.640983    1778 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: jpeg::Uncompress failed. Invalid JPEG data or crop win

Images: 4,024 | train: 3,219 | validation: 402 | test: 403


/tmp/ipykernel_1778/1086553765.py:1: RuntimeWarning: Skipped 59 unreadable image file(s) from the manifest.
  frame = load_manifest()


,count
label,
NO_FRACTURE,3307
ONE_FRACTURE,546
MULTIPLE_FRACTURES,171


In [3]:
def make_dataset(dataframe, shuffle=False):
    paths = dataframe['path'].to_numpy()
    labels = dataframe['label_index'].to_numpy(dtype=np.int32)
    def load(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        return image, label
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train, shuffle=True)
validation_dataset = make_dataset(validation)
test_dataset = make_dataset(test)
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train['label_index'].to_numpy())
class_weights = {index: float(weight) for index, weight in enumerate(weights)}
print('Class weights:', class_weights)

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.
Class weights: {0: 0.4056710775047259, 1: 2.4553775743707096, 2: 7.8321167883211675}


## Train each model separately

Each loop iteration creates a fresh model, trains it from the same training split, selects its best validation checkpoint, and evaluates it only once on the held-out test split. ImageNet weights are used only by the two transfer-learning models.

In [4]:
def create_model(model_name):
    if model_name == 'custom_cnn':
        return build_model()
    weights = 'imagenet' if USE_IMAGENET_WEIGHTS else None
    return build_transfer_model(model_name, weights=weights)

model_paths = {}
histories = {}
results = []

for model_name in MODEL_NAMES:
    print(f'\n===== Training {model_name} =====')
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = create_model(model_name)
    path = MODEL_DIR / f'{model_name}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(path, monitor='val_accuracy', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
    ]
    history = model.fit(train_dataset, validation_data=validation_dataset, epochs=MODEL_EPOCHS, class_weight=class_weights, callbacks=callbacks)
    histories[model_name] = history.history
    model_paths[model_name] = path
    del model


===== Training custom_cnn =====
Epoch 1/15


/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
E0000 00:00:1787654891.998720    1778 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/fracatlas_fracture_classifier_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1787654892.785776    1877 cuda_dnn.cc:461] Loaded cuDNN version 92400


    101/Unknown 18s 106ms/step - accuracy: 0.4324 - loss: 1.1140

I0000 00:00:1787654908.832243    1965 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 9556959603845375303
I0000 00:00:1787654908.832350    1965 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 10839622244544980631
I0000 00:00:1787654908.832359    1965 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 8135631178450555202
I0000 00:00:1787654908.832368    1965 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 6314292383459566780
I0000 00:00:1787654908.832375    1965 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 7012125144250404042
/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  se

101/101 ━━━━━━━━━━━━━━━━━━━━ 20s 120ms/step - accuracy: 0.4324 - loss: 1.1140 - val_accuracy: 0.8234 - val_loss: 0.7668 - learning_rate: 0.0010
Epoch 2/15


...

101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 96ms/step - accuracy: 0.7723 - loss: 0.5474 - val_accuracy: 0.8234 - val_loss: 0.4321 - learning_rate: 3.0000e-04
Epoch 9/15
  2/101 ━━━━━━━━━━━━━━━━━━━━ 7s 81ms/step - accuracy: 0.9219 - loss: 0.4173  

W0000 00:00:1787655228.637601    2959 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 19267840 bytes after encountering the first element of size 19267840 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


101/101 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - accuracy: 0.7906 - loss: 0.5043 - val_accuracy: 0.8234 - val_loss: 0.4309 - learning_rate: 9.0000e-05
Epoch 10/15
  2/101 ━━━━━━━━━━━━━━━━━━━━ 9s 95ms/step - accuracy: 0.7812 - loss: 0.5073  

W0000 00:00:1787655237.350730    2959 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 19267840 bytes after encountering the first element of size 19267840 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


101/101 ━━━━━━━━━━━━━━━━━━━━ 9s 89ms/step - accuracy: 0.7735 - loss: 0.5265 - val_accuracy: 0.8134 - val_loss: 0.4425 - learning_rate: 9.0000e-05


In [5]:
for model_name, path in model_paths.items():
    model = tf.keras.models.load_model(path)
    probabilities = model.predict(test_dataset, verbose=0)
    predicted = probabilities.argmax(axis=1)
    actual = np.concatenate([labels.numpy() for _, labels in test_dataset], axis=0)
    metrics = calculate_classification_metrics(actual, predicted)
    results.append({'model': model_name, **metrics})
    print(f'\n{model_name}')
    print(classification_report(actual, predicted, target_names=CLASS_NAMES, zero_division=0))

results_df = pd.DataFrame(results).sort_values([SELECTION_METRIC, 'macro_f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)
display(results_df.style.format({column: '{:.4f}' for column in results_df.columns if column != 'model'}))


custom_cnn
                    precision    recall  f1-score   support

       NO_FRACTURE       0.82      1.00      0.90       331
      ONE_FRACTURE       0.00      0.00      0.00        55
MULTIPLE_FRACTURES       0.00      0.00      0.00        17

          accuracy                           0.82       403
         macro avg       0.27      0.33      0.30       403
      weighted avg       0.67      0.82      0.74       403



/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



mobilenetv2
                    precision    recall  f1-score   support

       NO_FRACTURE       0.89      0.90      0.89       331
      ONE_FRACTURE       0.37      0.27      0.31        55
MULTIPLE_FRACTURES       0.31      0.47      0.37        17

          accuracy                           0.80       403
         macro avg       0.52      0.55      0.53       403
      weighted avg       0.79      0.80      0.79       403



/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



efficientnetb0
                    precision    recall  f1-score   support

       NO_FRACTURE       0.91      0.90      0.90       331
      ONE_FRACTURE       0.41      0.44      0.42        55
MULTIPLE_FRACTURES       0.41      0.41      0.41        17

          accuracy                           0.81       403
         macro avg       0.58      0.58      0.58       403
      weighted avg       0.82      0.81      0.82       403



,model,accuracy,macro_precision,macro_recall,macro_f1
0,efficientnetb0,0.8139,0.5770,0.5818,0.5793
1,mobilenetv2,0.7965,0.5202,0.5479,0.5260
2,custom_cnn,0.8213,0.2738,0.3333,0.3006


In [6]:
best_result = select_best_result(results)
best_name = best_result['model']
best_path = model_paths[best_name]
best_model = tf.keras.models.load_model(best_path)
best_model.save(ARTIFACT_DIR / 'fracture_classifier.keras')
results_df.to_csv(ARTIFACT_DIR / 'model_comparison.csv', index=False)
test.to_csv(ARTIFACT_DIR / 'test_manifest.csv', index=False)
metadata = {
    'modelVersion': f'fracatlas-{best_name}-1.0.0',
    'selectedModel': best_name,
    'classes': list(CLASS_NAMES),
    'imageSize': list(IMAGE_SIZE),
    'datasetCsv': str(DATASET_CSV),
    'trainCount': int(len(train)),
    'validationCount': int(len(validation)),
    'testCount': int(len(test)),
    'selectionMetric': SELECTION_METRIC,
    'comparison': results_df.to_dict(orient='records'),
}
(ARTIFACT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2, default=float), encoding='utf-8')
print(f'Selected {best_name} using {SELECTION_METRIC} (macro F1 tie-breaker) and saved it to {ARTIFACT_DIR / "fracture_classifier.keras"}')
display(results_df)

Selected efficientnetb0 using macro F1 and saved it to /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/ai-service/artifacts/fracture_classifier.keras


,model,accuracy,macro_precision,macro_recall,macro_f1
0,efficientnetb0,0.813896,0.577015,0.581803,0.579303
1,mobilenetv2,0.796526,0.520150,0.547873,0.526049
2,custom_cnn,0.821340,0.273780,0.333333,0.300636
